# 04 — Sensitivity Analysis and Statistical Inference

This notebook presents Sobol global sensitivity results, bootstrap/Bayesian inference,
and Decision Curve Analysis in detail.

In [1]:
import pandas as pd
import json
from pathlib import Path
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

## Sobol Global Sensitivity Analysis

Total-order indices (ST) quantify each parameter's contribution to outcome variance,
including interaction effects.

In [2]:
outcomes = ['unsafe_detection_rate', 'safe_throughput', 'false_negative_harm', 'mean_total_friction']
for outcome in outcomes:
    df = pd.read_csv(f'outputs/tables/sobol_{outcome}.csv')
    print(f'\n{outcome.replace("_", " ").title()}')
    print(df[['parameter', 'ST', 'status', 'rank']].to_string(index=False))


Unsafe Detection Rate
         parameter       ST     status  rank
       safety_gate 1.000000 configured     1
  unsafe_base_rate 0.679155    ASSUMED     2
         bias_gate 0.486024 configured     3
  calibration_gate 0.342183 configured     4
     evidence_gate 0.313528 configured     5
 traceability_gate 0.125810 configured     6
auditability_noise 0.011527    ASSUMED     7

Safe Throughput
         parameter       ST     status  rank
  calibration_gate 0.318143 configured     1
     evidence_gate 0.297365 configured     2
         bias_gate 0.210395 configured     3
       safety_gate 0.141609 configured     4
 traceability_gate 0.123777 configured     5
auditability_noise 0.016299    ASSUMED     6
  unsafe_base_rate 0.013375    ASSUMED     7

False Negative Harm
         parameter       ST     status  rank
       safety_gate 1.000000 configured     1
  unsafe_base_rate 1.000000    ASSUMED     2
         bias_gate 0.682354 configured     3
  calibration_gate 0.347061 configured 

## Statistical Inference

Bootstrap confidence intervals (500 resamples) and Bayesian Beta-posterior credible intervals
for the moderate threshold profile across replicates.

In [3]:
inf = json.loads(Path('outputs/processed/inference_results.json').read_text())
print(f"Detection: {inf['detection_mean']:.4f} [{inf['detection_ci_lower']:.4f}, {inf['detection_ci_upper']:.4f}]")
print(f"Throughput: {inf['throughput_mean']:.4f} [{inf['throughput_ci_lower']:.4f}, {inf['throughput_ci_upper']:.4f}]")
print(f"FN Harm: {inf['harm_mean']:.2f} [{inf['harm_ci_lower']:.2f}, {inf['harm_ci_upper']:.2f}]")
print(f"Friction: {inf['friction_mean']:.4f} [{inf['friction_ci_lower']:.4f}, {inf['friction_ci_upper']:.4f}]")
print(f"\nBayesian detection: {inf['detection_bayesian']['posterior_mean']:.4f} [{inf['detection_bayesian']['ci_lower']:.4f}, {inf['detection_bayesian']['ci_upper']:.4f}]")
print(f"Bayesian throughput: {inf['throughput_bayesian']['posterior_mean']:.4f} [{inf['throughput_bayesian']['ci_lower']:.4f}, {inf['throughput_bayesian']['ci_upper']:.4f}]")

Detection: 0.9957 [0.9938, 0.9976]
Throughput: 0.2099 [0.2027, 0.2173]
FN Harm: 6.72 [3.77, 9.99]
Friction: 3.7652 [3.7633, 3.7673]

Bayesian detection: 0.9957 [0.9948, 0.9965]
Bayesian throughput: 0.2099 [0.2049, 0.2150]


## Scenario Variation

Profile performance across 5 scenario families.

In [4]:
scenarios = pd.read_csv('outputs/raw/scenario_results.csv')
for scenario_idx in sorted(scenarios['scenario_idx'].unique()):
    s = scenarios[scenarios['scenario_idx'] == scenario_idx].iloc[0]
    print(f"\nScenario {scenario_idx + 1}: UBR={s['unsafe_base_rate']}, noise={s['auditability_noise']}, evidence={s['evidence_regime']}, drift={s['drift_regime']}")
    for profile in ['permissive', 'moderate', 'strict', 'very_strict']:
        p = scenarios[(scenarios['scenario_idx'] == scenario_idx) & (scenarios['profile_name'] == profile)]
        if len(p) > 0:
            print(f"  {profile:12s}: det={p['unsafe_detection_rate'].mean():.3f}, thr={p['safe_throughput'].mean():.3f}, harm={p['false_negative_harm'].mean():.1f}")


Scenario 1: UBR=0.1, noise=0.05, evidence=mixed, drift=medium
  permissive  : det=0.940, thr=0.495, harm=53.0
  moderate    : det=1.000, thr=0.192, harm=0.0
  strict      : det=1.000, thr=0.021, harm=0.0
  very_strict : det=1.000, thr=0.003, harm=0.0

Scenario 2: UBR=0.2, noise=0.05, evidence=mixed, drift=medium
  permissive  : det=0.844, thr=0.520, harm=231.6
  moderate    : det=0.999, thr=0.213, harm=1.8
  strict      : det=1.000, thr=0.023, harm=0.0
  very_strict : det=1.000, thr=0.004, harm=0.0

Scenario 3: UBR=0.3, noise=0.05, evidence=mixed, drift=medium
  permissive  : det=0.767, thr=0.547, harm=454.6
  moderate    : det=0.988, thr=0.239, harm=20.1
  strict      : det=1.000, thr=0.026, harm=0.0
  very_strict : det=1.000, thr=0.004, harm=0.0

Scenario 4: UBR=0.2, noise=0.02, evidence=artefact-heavy, drift=low
  permissive  : det=0.832, thr=0.543, harm=246.0
  moderate    : det=1.000, thr=0.219, harm=0.2
  strict      : det=1.000, thr=0.025, harm=0.0
  very_strict : det=1.000, th

## Sensitivity Assertions

In [5]:
# Verify Sobol key claims
sobol_det = pd.read_csv('outputs/tables/sobol_unsafe_detection_rate.csv')
sobol_harm = pd.read_csv('outputs/tables/sobol_false_negative_harm.csv')
sobol_thr = pd.read_csv('outputs/tables/sobol_safe_throughput.csv')
sobol_fric = pd.read_csv('outputs/tables/sobol_mean_total_friction.csv')

assert float(sobol_det.loc[sobol_det['parameter'] == 'safety_gate', 'ST'].iloc[0]) == 1.0, 'Safety ST for detection != 1.0'
assert float(sobol_harm.loc[sobol_harm['parameter'] == 'safety_gate', 'ST'].iloc[0]) == 1.0, 'Safety ST for harm != 1.0'
assert abs(float(sobol_thr.loc[sobol_thr['parameter'] == 'calibration_gate', 'ST'].iloc[0]) - 0.32) < 0.02, 'Calibration ST for throughput != ~0.32'
assert abs(float(sobol_fric.loc[sobol_fric['parameter'] == 'safety_gate', 'ST'].iloc[0]) - 0.23) < 0.02, 'Safety ST for friction != ~0.23'

# Verify inference claims
assert abs(inf['detection_mean'] - 0.996) < 0.002, f"Detection mean {inf['detection_mean']} not ~0.996"
assert abs(inf['throughput_mean'] - 0.210) < 0.002, f"Throughput mean {inf['throughput_mean']} not ~0.210"

print('All sensitivity and inference assertions PASSED')

All sensitivity and inference assertions PASSED
